In [ ]:
// 1. импорты для сессий и спарк sql
import $ivy.`org.apache.spark::spark-core:3.5.6`
import $ivy.`org.apache.spark::spark-sql:3.5.6`
import org.apache.spark.{SparkConf, SparkContext}
import org.apache.spark.sql.{DataFrame, SparkSession, SaveMode}

// 2. стоп сессий
SparkSession.getActiveSession.foreach(_.stop())

// 3. Конфиг и создание сессий
val conf = new SparkConf()
  .setAppName("Dataframe")
  .setMaster("local[*]")
  .set("spark.driver.memory","1g")
  .set("spark.log.level", "WARN")

val spark = SparkSession.builder().config(conf).getOrCreate()
val sc = spark.sparkContext
println(s"Spark version: ${spark.version}")

// 4. методы для преобразования скала коллекций в датафреймы 
import spark.implicits._

// 5. поддержка типов данных спарка, которыми оперирует Dataframe
import org.apache.spark.sql.types._

// 6. определение схемы для json файла
val jsonSchema = StructType(Array(
    StructField("name", StructType(Array(
        StructField("common", StringType, true),
        StructField("official", StringType, true),
        StructField("native", MapType(StringType, StructType(Array(
            StructField("official", StringType, true),
            StructField("common", StringType, true)
        ))), true)
    )), true),
    StructField("tld", ArrayType(StringType, true), true),
    StructField("cca2", StringType, true),
    StructField("ccn3", StringType, true),
    StructField("cca3", StringType, true),
    StructField("cioc", StringType, true),
    StructField("independent", BooleanType, true),
    StructField("status", StringType, true),
    StructField("unMember", BooleanType, true),
    StructField("currencies", MapType(StringType, StructType(Array(
        StructField("name", StringType, true),
        StructField("symbol", StringType, true)
    ))), true),
    StructField("idd", StructType(Array(
        StructField("root", StringType, true),
        StructField("suffixes", ArrayType(StringType, true), true)
    )), true),
    StructField("capital", ArrayType(StringType, true), true),
    StructField("altSpellings", ArrayType(StringType, true), true),
    StructField("region", StringType, true),
    StructField("subregion", StringType, true),
    StructField("languages", MapType(StringType, StringType, true), true),
    StructField("translations", MapType(StringType, StructType(Array(
        StructField("official", StringType, true),
        StructField("common", StringType, true)
    ))), true),
    StructField("latlng", ArrayType(DoubleType, true), true),
    StructField("landlocked", BooleanType, true),
    StructField("borders", ArrayType(StringType, true), true),
    StructField("area", DoubleType, true),
    StructField("flag", StringType, true),
    StructField("demonyms", MapType(StringType, StructType(Array(
        StructField("f", StringType, true),
        StructField("m", StringType, true)
    ))), true)
))

// 7. чтение json файла в Dataframe 
val df = spark.read
            .format("json")
            .schema(jsonSchema)
            .option("multiLine", "true")
            .option("mode", "FAILFAST")
            .load("countries.json")

// смотрим схему
//  println(df.printSchema)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/12/15 16:29:38 WARN Utils: Your hostname, ubuntusd resolves to a loopback address: 127.0.1.1; using 192.168.1.169 instead (on interface ens192)
25/12/15 16:29:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/12/15 16:29:38 INFO SparkContext: Running Spark version 3.5.6
25/12/15 16:29:38 INFO SparkContext: OS info Linux, 5.4.0-216-generic, amd64
25/12/15 16:29:38 INFO SparkContext: Java version 11.0.27
25/12/15 16:29:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting Spark log level to "WARN".


Spark version: 3.5.6


import $ivy.$
import $ivy.$
import org.apache.spark.{SparkConf, SparkContext}
import org.apache.spark.sql.{DataFrame, SparkSession, SaveMode}
conf: SparkConf = org.apache.spark.SparkConf@100fc2ff
spark: SparkSession = org.apache.spark.sql.SparkSession@75f2debd
sc: SparkContext = org.apache.spark.SparkContext@206389d6
import spark.implicits._
import org.apache.spark.sql.types._
jsonSchema: StructType = StructType(
  StructField(
    "name",
    StructType(
      StructField("common", StringType, true, {}),
      StructField("official", StringType, true, {}),
      StructField(
        "native",
        MapType(
          StringType,
          StructType(
            StructField("official", StringType, true, {}),
            StructField("common", StringType, true, {})
          ),
          true
        ),
        true,
        {}
      )
    ),
    true,
    {}
  ),
  StructField("tld", ArrayType(StringType, true), true, {}),
  StructField("cca2", StringType, true, {}),
  StructField("c

In [ ]:
import org.apache.spark.sql.functions.{col, size, coalesce, array, concat_ws}

// 8. Функция, которая возвращает датафрейм со странами, которые граничат с 5 или более чем с 5 странами. 
// В датафрейме должны быть столбцы:
// Country - Международное название страны
// NumBorders - Количество граничащих стран
// BorderCountries - Список граничащих стран в формате строки через запятую.
// def countryList(df: DataFrame): Unit = {
def countryList(df: DataFrame): DataFrame = {
    df.select(
        col("name.official").as("Country"), 
        size(coalesce(col("borders"), array())).as("NumBorders"),
        concat_ws(", ", coalesce(col("borders"), array())).as("BorderCountries")
    )
        .filter(col("NumBorders") >= 5)
        .sort("Country")
    }

// 9. Запись результата работы ф-и в файлы 
val resultСountry = countryList(df)
resultСountry.write
    .mode(SaveMode.Overwrite)
    .parquet("countryList")
// resultСountry.show(truncate=false)



import org.apache.spark.sql.functions.{col, size, coalesce, array, concat_ws}
defined function countryList
resultСountry: DataFrame = [Country: string, NumBorders: int ... 1 more field]

In [7]:
import org.apache.spark.sql.functions.{explode, count, lit, collect_list}

// 10. Функция, которая возвращает рейтинг языков, на которых говорят в наибольшем количестве стран. 
// В датафрейме должны быть столбцы:
// Language - название языка;
// NumCountries - количество стран, в которых говорят на языке;
// Countries - список международных названий стран, в которых говорят на языке, в формате `ArrayType
def languageList(df: DataFrame): DataFrame = {
    df.select(
        col("name.official").as("Country"), 
        explode(col("languages")).as(Seq("lang_code","Language"))
    ).groupBy("Language")
    .agg(
        count(lit(1)).as("NumCountries"),
        collect_list(col("Country")).as("Countries")
    )
    .orderBy(col("NumCountries").desc)
}

// 9. Запись результата работы ф-и в файлы 
val resultLanguage = languageList(df)
resultLanguage.write
    .mode(SaveMode.Overwrite)
    .parquet("languageList")

// resultLanguage.show(truncate=false)

import org.apache.spark.sql.functions.{explode, count, lit, collect_list}
defined function languageList
resultLanguage: DataFrame = [Language: string, NumCountries: bigint ... 1 more field]